# *CoNNear*: A convolutional neural-network model of human cochlear mechanics and filter tuning for real-time applications

Python notebook for reproducing the evaluation results of the proposed CoNNear model.

## Prerequisites

- First, let us compile the cochlea_utils.c file that is used for solving the transmission line (TL) model of the cochlea. This requires some C++ compiler which should be installed beforehand. Then go the connear folder from the terminal and run:
```  
gcc -shared -fpic -O3 -ffast-math -o tridiag.so cochlea_utils.c  
```
- Install numpy, scipy, PyTorch

## Import required python packages and functions
Import required python packages and load the connear model.

In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import torch
from connear_pytorch import load_connear

from tlmodel.get_tl_vbm_and_oae import tl_vbm_and_oae

connear = load_connear("connear/Gmodel.pt")
connear.summary()

Define some functions here


In [ ]:
def rms(x):
    # compute rms of a matrix
    sq = np.mean(np.square(x), axis=0)
    return np.sqrt(sq)

In [ ]:
# Define model specific variables
down_rate = 2
fs = 20e3
fs_tl = 100e3
p0 = 2e-5
factor_fs = int(fs_tl / fs)
right_context = 256
left_context = 256
# load CFs
CF = np.loadtxt("tlmodel/cf.txt")

## Click response
Compare the responses of the models to a click stimulus.      
**Notice that for all the simulations, TL model operates at 100kHz and the CoNNear model operates at 20kHz.**

In [ ]:
# Define the click stimulus
dur = 128.0e-3  # for 2560 samples #CONTEXT
click_duration = 2  # 100 us click
stim = np.zeros((1, int(dur * fs)))
L = 70.0
samples = dur * fs
click_duration = 2  # 100 us click
click_duration_tl = factor_fs * click_duration
silence = 60  # samples in silence
samples = int(samples - right_context - left_context)
"""
# GET TL model response
stim = np.zeros((1, (samples + right_context + left_context)*factor_fs))
stim[0, (factor_fs * (right_context+silence)) : (factor_fs * (right_context+silence)) + click_duration_tl] = 2 * np.sqrt(2) * p0 * 10**(L/20)

output = tl_vbm_and_oae(stim , L)
CF = output[0]['cf'][::down_rate]
# basilar membrane motion for click response
# the context samples (first and last 256 samples)
# are removed. Also downsample it to 20kHz
bmm_click_out_full = np.array(output[0]['v'])
stimrange = range(right_context*factor_fs, (right_context*factor_fs) + (factor_fs*samples))
bmm_click_tl = sp_sig.resample_poly(output[0]['v'][stimrange,::down_rate], fs, fs_tl)
bmm_click_tl = bmm_click_tl.T 
"""
# Prepare the same for CoNNear model
stim = np.zeros((1, int(dur * fs)))
stim[0, right_context + silence : right_context + silence + click_duration] = (
    2 * np.sqrt(2) * p0 * 10 ** (L / 20)
)
# Get the CoNNear response
stim = np.expand_dims(stim, axis=2)
connear_pred_click = connear.predict(stim.T, verbose=1)
bmm_click_connear = connear_pred_click[0, :, :].T * 1e-6

Plotting the results.

In [ ]:
plt.plot(stim[0, 256:-256]), plt.xlim(0, 2000)
plt.show()
"""
plt.imshow(bmm_click_tl, aspect='auto', cmap='jet')
plt.xlim(0,2000), plt.clim(-4e-7,5e-7)
plt.colorbar()
plt.show()
"""
plt.imshow(bmm_click_connear, aspect="auto", cmap="jet")
plt.xlim(0, 2000), plt.clim(-4e-7, 5e-7)
plt.colorbar()
plt.show()

## Cochlear Excitation Patterns
Here, we plot the simulated RMS levels of basilar memberane (BM) displacement across CF for tone stimuli presented at SPLs between 0 and 90 dB SPL.

In [ ]:
f_tone = (
    1e3  # You can change this tone frequency to see how the excitation pattern changes
)
# with stimulus frequency
fs = 20e3
p0 = 2e-5
dur = 102.4e-3  # for 2048 samples
window_len = int(fs * dur)
L = np.arange(0.0, 91.0, 10.0)  # SPLs from 0 to 90dB

# CoNNear
t = np.arange(0.0, dur, 1.0 / fs)
hanlength = int(10e-3 * fs)  # 10ms length hanning window
stim_sin = np.sin(2 * np.pi * f_tone * t)
han = signal.windows.hann(hanlength)
stim_sin[: int(hanlength / 2)] = (
    stim_sin[: int(hanlength / 2)] * han[: int(hanlength / 2)]
)
stim_sin[-int(hanlength / 2) :] = (
    stim_sin[-int(hanlength / 2) :] * han[int(hanlength / 2) :]
)
stim = np.zeros((len(L), int(len(stim_sin))))
# total_length = 2560 #CONTEXT
total_length = window_len + right_context + left_context  #  CONTEXT
stim = np.zeros((len(L), total_length))  # CONTEXT
for j in range(len(L)):
    stim[j, right_context : window_len + right_context] = (
        p0 * np.sqrt(2) * 10 ** (L[j] / 20) * stim_sin
    )

# prepare for feeding to the DNN
stim = np.expand_dims(stim, axis=2)

connear_pred_tone = connear.predict(stim, verbose=1)
bmm_tone_connear = connear_pred_tone  # * 1e-6
bmm_tone_connear.shape
# Compute rms for each level
cochlear_pred_tone_rms = np.vstack([rms(bmm_tone_connear[i]) for i in range(len(L))])

# Plot the RMS
cftile = np.tile(CF, (len(L), 1))
plt.semilogx(cftile.T, 20.0 * np.log10(cochlear_pred_tone_rms.T))
(
    plt.xlim(0.25, 8.0),
    plt.grid(which="both"),
)
plt.xticks(
    ticks=(0.25, 0.5, 1.0, 2.0, 4.0, 8.0), labels=(0.25, 0.5, 1.0, 2.0, 4.0, 8.0)
)
plt.ylim(-80, 20)
plt.xlabel("CF (kHz)")
plt.ylabel("RMS of y_bm (dB)")
plt.title("CoNNear Predicted")
plt.show()